# Expfit notes 2: Single exponential error and fit

Here, we work out the error function ($E$) for a single exponential fit, along with its Jacobian ($J$) and Hessian ($H$).
Using these, we then explore a simple Levenberg-Marquardt (LM) optimisation scheme.

We will use the mean squared error (MSE), as it has an easily derived $J$ and $H$.

\begin{align}
E &= \frac{1}{n} \sum \left(a - v_i + b e^{-t_i / \tau} \right)^2
\end{align}

To simplify the notation, we define
\begin{align}
e_i &= e^{-t_i / \tau}
\end{align}
and
\begin{align}
f_i = a - v_i + b e_i
\end{align}

## Jacobian

To find the Jacobian, we use the chain rule
\begin{align}
\frac{\partial}{\partial x} f(x)^2 = 2 f(x) \frac{\partial}{\partial x} f(x)
\qquad \rightarrow \qquad
\frac{\partial}{\partial x} f_i^2 = 2 f_i \frac{\partial}{\partial x} f_i
\end{align}
to derive
\begin{align}
\frac{\partial}{\partial a} f_i &= 1
    & \rightarrow &&
    \frac{\partial E}{\partial a} &= \frac{2}{n} \sum f_i \\
\frac{\partial}{\partial b} f_i &= e_i
    & \rightarrow &&
    \frac{\partial E}{\partial b} &= \frac{2}{n} \sum f_i e_i \\
\frac{\partial}{\partial \tau} f_i &= \frac{b t_i}{\tau^2} e_i
    & \rightarrow &&
    \frac{\partial E}{\partial \tau} &= \frac{2b}{n\tau^2} \sum f_i e_i t_i
\end{align}

### Interpretation

### Calculation
#TODO

Computationally, this Jacobian has a useful form.

In a compiled language, we could write
```
for i in ...
    e = exp(c * x[i])
    da_increment = a - y[i] + b * e
    db_increment = da_increment * e
    dc_increment = db_increment * x[i]
```

In Python/NumPy, it's faster to implement this with array operations
```
e = np.exp(c * x)                  # 2 array operations
f = a - v + b * e                  # 3
g = e * f                          # 1
da = 2 / n * np.sum(f)             # 1
db = 2 / n * np.sum(g)             # 1 
dc = 2 / n * np.sum(g * x) * b     # 2 
```
which works out as 10 array operations.

### Checking our workings

We can check against finite differences:

In [1]:
import numpy as np

def f(t, a, b, tau):
    return a + b * np.exp(-t / tau)

In [3]:
a, b, tau = 5, 5, 0.5
n = 100
t = np.linspace(0, 1, n)
v = f(t, a, b, tau)

a0, b0, tau0 = a + 1, b - 1, tau + 0.1
v0 = f(t, a0, b0, tau0)

def mse(t, v, a, b, tau):
    return np.sum((a - v + b * np.exp(-t / tau))**2) / len(t)

def mse_jac(t, v, a, b, tau):
    m = 1 / n
    e = np.exp(-t / tau)
    g = a - v + b * e
    ti2 = 1 / (tau * tau)
    eg = e * g
    da = 2 * m * np.sum(g)
    db = 2 * m * np.sum(eg)
    dc = 2 * m * np.sum(eg * t) * b * ti2
    return m * np.sum(g * g), da, db, dc

def mse_jac_fd(t, v, a, b, tau, dp=1e-9):
    m = 1 / n
    r1 = mse(t, v, a, b, tau)
    da = (mse(t, v, a + dp, b, tau) - r1) / dp
    db = (mse(t, v, a, b + dp, tau) - r1) / dp
    dt = (mse(t, v, a, b, tau + dp) - r1) / dp    
    return r1, da, db, dt

a0 = a
b0 = b
m0 = mse(t, v, a0, b0, tau0)

m1, da1, db1, dt1 = mse_jac(t, v, a0, b0, tau0)
m2, da2, db2, dt2 = mse_jac_fd(t, v, a0, b0, tau0)

print(f'Plain mse {m0}')
print(f'  mse_jac {m1}')
print(f'  mse_fd  {m2}')
print(f'dE/da {da1}')
print(f'   fd {da2}')
print(f'dE/db {db1}')
print(f'   fd {db2}')
print(f'dE/dt {dt1}')
print(f'   fd {dt2}')

print(mse(t, v, a0, b0, tau))
print(mse(t, v, a0, b0, tau + 0.01))

Plain mse 0.07973060766522976
  mse_jac 0.07973060766522976
  mse_fd  0.07973060766522976
dE/da 0.5406338152554949
   fd 0.540633857126771
dE/db 0.2335398256494637
   fd 0.23353985412200015
dE/dt 1.4616965493853764
   fd 1.4616965138269933
1.167267620694216e-31
0.0009300053311887155


And even run a plain gradient descent:

In [11]:
alpha = 0.5
a1, b1, tau1 = a0, b0, tau0

for i in range(10):
    print(i, m1)
    print(f'  {a} {a1:6f}')
    print(f'  {b} {b1:6f}')
    print(f'  {tau} {tau1:6f}')
    
    m1, da, db, dtau = mse_jac(t, v, a1, b1, tau1)
    a1 -= da * alpha]
    b1 -= db * alpha
    tau1 -= dtau * alpha

print(i, m1)
print(f'  {a} {a1:6f}')
print(f'  {b} {b1:6f}')
print(f'  {tau} {tau1:6f}')

0 2204107639063.533
  5 5.000000
  5 5.000000
  0.5 0.600000
1 0.07973060766522976
  5 4.729683
  5 4.883230
  0.5 -0.130848
2 7241492.0456597
  5 -1362.583283
  5 -1483253.956730
  0.5 -397460041.800509
3 2204107561426.7837
  5 1483261.127098
  5 1369.755518
  0.5 -397460041.800516
4 2204107572517.748
  5 -1362.587019
  5 -1483253.960466
  0.5 -397460041.800516
5 2204107583608.7134
  5 1483261.130833
  5 1369.759253
  0.5 -397460041.800523
6 2204107594699.677
  5 -1362.590754
  5 -1483253.964201
  0.5 -397460041.800523
7 2204107605790.641
  5 1483261.134568
  5 1369.762989
  0.5 -397460041.800530
8 2204107616881.6045
  5 -1362.594489
  5 -1483253.967936
  0.5 -397460041.800530
9 2204107627972.5684
  5 1483261.138303
  5 1369.766724
  0.5 -397460041.800537
9 2204107639063.533
  5 -1362.598224
  5 -1483253.971671
  0.5 -397460041.800537


## Hessian

We can go a step further and find the Hessian

From $\frac{\partial E}{\partial a} = \frac{2}{n} \sum f_i $
\begin{align}
\frac{\partial^2 E}{\partial a^2} 
    &= \frac{2}{n} \sum 1 = 2 \\
\frac{\partial^2 E}{\partial b \, \partial a} 
    &= \frac{2}{n} \sum e_i \\
\frac{\partial^2 E}{\partial \tau \, \partial a} 
    &= -\frac{2b}{n\tau^2} \sum t_i e_i
\end{align}

From $\frac{\partial E}{\partial b} = \frac{2}{n} \sum f_i e_i $
\begin{align}
\frac{\partial^2 E}{\partial a \, \partial b} 
    &= \frac{2}{n} \sum e_i \\
\frac{\partial^2 E}{\partial b^2}
    &= \frac{2}{n} \sum e_i^2 \\
\frac{\partial^2 E}{\partial c \, \partial b}
    &= \frac{2}{n} \sum (f_i + b e_i) x_i e_i
\end{align}

From $\frac{\partial E}{\partial \tau} = -\frac{2b}{n\tau^2} \sum f_i x_i e_i $
\begin{align}
\frac{\partial^2 E}{\partial a \, \partial c}
    &= \frac{2b}{n} \sum x_i e_i \\
\frac{\partial^2 E}{\partial b \, \partial c}
    &= \frac{2}{n} \sum (f_i + b e_i) x_i e_i   \\
\frac{\partial^2 E}{\partial c^2}
    &= \frac{2b}{n} \sum (f_i + b e_i) x_i^2 e_i \\
\end{align}

**Sanity checks**: 
\begin{align}
\frac{\partial^2 E}{\partial a \, \partial b} &= \frac{\partial^2 E}{\partial b \, \partial a}, &&
\frac{\partial^2 E}{\partial a \, \partial c} &= \frac{\partial^2 E}{\partial c \, \partial a}, &&
\frac{\partial^2 E}{\partial b \, \partial c} &= \frac{\partial^2 E}{\partial c \, \partial b} 
\end{align}

### Checking our workings

In [4]:
def mse_jac_hes(x, y, p):
    a, b, c = p
    m = 1 / n
    e = np.exp(c * x)
    f = a - y + b * e
    ef = e * f
    mse = m * np.sum(f * f)

    # Jacobian
    jac = np.array([
        2 * m * np.sum(f),
        2 * m * np.sum(ef),
        2 * m * np.sum(ef * x) * b
    ])

    # Hessian
    ex = e * x
    aex = (f + b * e) * ex
    hes = np.array([
        [2, 2 * m * np.sum(e), 2 * b * m * np.sum(ex)],
        [0, 2 * m * np.sum(e * e), 2 * m * np.sum(aex)],
        [0, 0, 2 * m * b * np.sum(x * aex)],
    ])
    hes[1, 0] = hes[0, 1]
    hes[2, 0] = hes[0, 2]
    hes[2, 1] = hes[1, 2]
    
    return mse, jac, hes

m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, c0))

print(m1)
print(j1)
print(h1)

NameError: name 'x' is not defined

In [ ]:
def mse_jac_hes_fd(x, y, p, dp=1e-6):
    a, b, c = p
    err = mse(x, y, a, b, c)
    da = lambda a, b, c: (mse(x, y, a + dp, b, c) - mse(x, y, a, b, c)) / dp
    db = lambda a, b, c: (mse(x, y, a, b + dp, c) - mse(x, y, a, b, c)) / dp
    dc = lambda a, b, c: (mse(x, y, a, b, c + dp) - mse(x, y, a, b, c)) / dp
    jac = np.array([da(a, b, c), db(a, b, c), dc(a, b, c)])
    hes = np.array([
        [(da(a + dp, b, c) - da(a, b, c)) / dp,
         (da(a, b + dp, c) - da(a, b, c)) / dp,
         (da(a, b, c + dp) - da(a, b, c)) / dp], 
        [(db(a + dp, b, c) - db(a, b, c)) / dp,
         (db(a, b + dp, c) - db(a, b, c)) / dp,
         (db(a, b, c + dp) - db(a, b, c)) / dp],
        [(dc(a + dp, b, c) - dc(a, b, c)) / dp,
         (dc(a, b + dp, c) - dc(a, b, c)) / dp,
         (dc(a, b, c + dp) - dc(a, b, c)) / dp]        
    ])
    
    return err, jac, hes

m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, c0))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, c0))
print(f'Plain mse {m0}')
print(f'  mse_jac {m1}')
print(f'  mse_fd  {m2}')
print()
print(f'dE/da {j1[0]}')
print(f'   fd {j2[0]}')
print(f'dE/db {j1[1]}')
print(f'   fd {j2[1]}')
print(f'dE/dc {j1[2]}')
print(f'   fd {j2[2]}')
print()
print(h1)
print()
print(h2)

## Optimisation

Now that we have a Jacobian and Hessian, we can try classic optimisation methods.

### Newton's method

First, we try a plain Newton's method:
\begin{align}
p_{i+1} = p_i - H(p_i)^{-1} J(p_i)
\end{align}

In [ ]:
ptrue = np.array([a, b, c])
p0 = np.array([a0, b0, c0], dtype=float)
m1, j1, h1 = mse_jac_hes(x, y, p0)
p1 = p0 - np.linalg.inv(h1) @ j1

print(f'True     {ptrue}')
print(f'Initial  {p0}')
print(f'One step {p1}')

This isn't promising, but we can try a few more:

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
for i in range(10):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.inv(h1) @ j1
print(f'Solution: {p1}')
print(f'Error: {m1}')

So with multiple steps we get very close, very quickly!

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
for i in range(20):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= np.linalg.inv(h1) @ j1
print(f'Solution: {p1}')
print(f'Error: {m1}')

But with more steps it gets worse.
We can fix this with a damping factor:

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
for i in range(20):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.inv(h1) @ j1
print(f'Solution: {p1}')
print(f'Error: {m1}')

### Comparison with Scipy

To get an idea of what "good performance" looks like, we can compare with SciPy, which uses BFGS: a method that _approximates_ the Hessian.

Unfortunately it expects seperate functions for error and jacobian, so we need to implement something like this:

In [ ]:
class CachedHesJac:
    _p = None
    _mse = None
    _jac = None
    _hes = None
    def _do(self, p):
        if np.any(p != self._p):
            self._mse, self._jac, self._hes = mse_jac_hes(x, y, p)
            self._p = np.copy(p)
    def mse(self, p):
        self._do(p)
        return self._mse
    def jac(self, p):
        self._do(p)
        return self._jac
    def hes(self, p):
        self._do(p)
        return self._hes

which will avoid repeated calculations and make the comparison a bit fairer.

The [default method in scipy is BFGS](https://github.com/scipy/scipy/blob/8c75ae75176236f233824e9a0483c26a69e6dfec/scipy/optimize/_minimize.py#L606), and the [default stopping criterion is a tolerance on the norm of the gradient](https://github.com/scipy/scipy/blob/8c75ae75176236f233824e9a0483c26a69e6dfec/scipy/optimize/_minimize.py#L679) with [default value](https://github.com/scipy/scipy/blob/8c75ae75176236f233824e9a0483c26a69e6dfec/scipy/optimize/_optimize.py#L1346) 1e-5

In [ ]:
import timeit
from scipy.optimize import minimize as fmin

chj = CachedHesJac()

time = timeit.default_timer()
r = fmin(chj.mse, p0, jac=chj.jac, method='bfgs', tol=1e-5)
time = timeit.default_timer() - time
print(r)
print(f'Time: {time}')

Without Jacobian:

In [ ]:
time = timeit.default_timer()
r = fmin(chj.mse, p0, method='bfgs', tol=1e-5)
time = timeit.default_timer() - time
print(r)
print(f'Time: {time}')

Newton's method for the same number of iterations:

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
time = timeit.default_timer()
for i in range(14):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.inv(h1) @ j1
time = timeit.default_timer() - time
print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Time: {time}')

This is promising.

It may be faster to use "solve" instead of inverting and multiplying, but we can't tell with this benchmark:

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
time = timeit.default_timer()
for i in range(14):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.solve(h1, j1)
time = timeit.default_timer() - time
print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Time: {time}')

Finally, we can add in the gradient norm tolerance:

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
time = timeit.default_timer()
for i in range(1000):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    p1 -= 0.5 * np.linalg.solve(h1, j1)
    if np.linalg.norm(j1) < 1e-5:
        break    
time = timeit.default_timer() - time
print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Iterations: {i}')
print(f'Time: {time}')

### LM-style update

Newton:
\begin{align}
p_{i+1} = p_i - H(p_i)^{-1} J(p_i)
\end{align}

Levenberg:
\begin{align}
p_{i+1} = p_i - (H(p_i) + \lambda I)^{-1} J(p_i)
\end{align}

where usually $H$ is approximated by $J^\intercal J$ (sometime called the Gauss-Newton Hessian), but here we'll use the true Hessian.

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
print(f'Initial {p1}')

time = timeit.default_timer()
alpha = 1
m1 = m_last = mse(x, y, *p1[0])
eye = np.eye(3)
for i in range(1000):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    if np.linalg.norm(j1) < 1e-5:
        break    
    ps = p1 - np.linalg.solve(h1 + alpha * eye, j1)
    ms = mse(x, y, *ps[0])
    if ms < m1:
        alpha *= 0.1
        p1 = ps
    elif ms > m1:
        alpha *= 10
time = timeit.default_timer() - time

print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Iterations: {i}')
print(f'Time: {time}')

This is _fast_.

Levenberg:
\begin{align}
p_{i+1} = p_i - (H(p_i) + \lambda I)^{-1} J(p_i)
\end{align}

Levenberg-Marquardt:
\begin{align}
p_{i+1} = p_i - (H(p_i) + \lambda \text{diag}(H))^{-1} J(p_i)
\end{align}

In [ ]:
p1 = np.array([[a0, b0, c0]], dtype=float)
print(f'Initial {p1}')

time = timeit.default_timer()
alpha = 1
m1 = m_last = mse(x, y, *p1[0])
eye = np.eye(3)
for i in range(1000):
    m1, j1, h1 = mse_jac_hes(x, y, p1[0])
    if np.linalg.norm(j1) < 1e-5:
        break
    ps = p1 - np.linalg.solve(h1 + alpha * eye * h1, j1)
    ms = mse(x, y, *ps[0])
    if ms < m1:
        alpha *= 0.1
        p1 = ps
    elif ms > m1:
        alpha *= 10
time = timeit.default_timer() - time

print(f'Solution: {p1}')
print(f'Error: {m1}')
print(f'Iterations: {i}')
print(f'Time: {time}')

Which shaves off another iteration for this test case.

### More tests

In [ ]:
def opt(p0):
    p1 = np.array([p0], dtype=float)
    print(f'Initial {p1}')
    
    time = timeit.default_timer()
    alpha = 1
    m1 = m_last = mse(x, y, *p1[0])
    eye = np.eye(3)
    for i in range(1000):
        m1, j1, h1 = mse_jac_hes(x, y, p1[0])
        if np.linalg.norm(j1) < 1e-5:
            break
        ps = p1 - np.linalg.solve(h1 + alpha * eye * h1, j1)
        ms = mse(x, y, *ps[0])
        if ms < m1:
            alpha *= 0.1
            p1 = ps
        elif ms > m1:
            alpha *= 10
    time = timeit.default_timer() - time
    
    print(f'Solution: {p1}')
    print(f'Norm: {np.linalg.norm(j1)}')
    print(f'Error: {m1}')
    print(f'Iterations: {i}')
    print(f'Time: {time}')

opt([1, 1, 1])

In [ ]:
opt([10, 10, 10])

In [ ]:
opt([-10, -10, -10])